# Same-split implementation: Pang MARS and Jose LightGBM

This Colab-ready notebook evaluates representative methods from:

- Pang (2025): MARS degree 1/2.
- Jose et al. (2024): LightGBM.

Both methods use the released prepared dataset, fixed split, OOF assignments,
and decision threshold. Published values are not used for model selection.

Open this notebook from the repository and choose **Runtime > Run all**.
Set `LICE_PROJECT_ROOT` only when automatic repository discovery fails.


## Step 1 - Mount Drive and locate the repository


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

configured_root = os.environ.get('LICE_PROJECT_ROOT')
PROJECT_CANDIDATES = [
    Path(configured_root) if configured_root else None,
    Path.cwd(),
    Path('/content/drive/MyDrive/Research/LICE_Guided_Model_Refinement_v1.3.0'),
    Path('/content/drive/MyDrive/LICE_Guided_Model_Refinement_v1.3.0'),
]
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in PROJECT_CANDIDATES
     if candidate and (candidate / 'data/diabetes_brfss2015_prepared.csv').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Repository not found. Set LICE_PROJECT_ROOT to the folder containing '
        'data/diabetes_brfss2015_prepared.csv and splits/split_assignments.csv.gz.'
    )

WORK_ROOT = PROJECT_ROOT
SOURCE_ROOT = PROJECT_ROOT
for folder in [
    'config/recent_comparators', 'intermediate/recent_comparators',
    'models/recent_comparators', 'predictions/recent_comparators',
    'results/recent_comparators', 'environment/recent_comparators',
]:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

DATA_FILE = PROJECT_ROOT / 'data/diabetes_brfss2015_prepared.csv'
SPLIT_FILE = PROJECT_ROOT / 'splits/split_assignments.csv.gz'
BASELINE_PREDICTIONS = PROJECT_ROOT / 'predictions/test/eight_configuration_test_predictions_wide.csv.gz'

print('Repository root:', PROJECT_ROOT)


## Step 2 - Install the two model packages

MARS is run with the maintained R `earth` package. The older Python
`sklearn-contrib-py-earth` package is deliberately not used. Keep Colab's
preinstalled NumPy/Pandas/SciPy/scikit-learn stack intact; replacing those
packages inside a live runtime can create binary/API incompatibilities.



In [ ]:
%pip -q install --no-deps lightgbm==4.6.0
!Rscript -e "if (!requireNamespace('earth', quietly=TRUE)) install.packages('earth', repos='https://cloud.r-project.org')"

print('Package installation complete. If this cell changed an installed package,')
print('use Runtime > Restart session once, then run the notebook again from Step 1.')



## Step 3 - Imports, seeds, checksums and frozen protocol



In [ ]:
import hashlib
import json
import os
import platform
import random
import subprocess
import sys
import time

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import scipy
import sklearn
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    cohen_kappa_score, confusion_matrix, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score,
)

SEED = 42
THRESHOLD = 0.50
N_FOLDS = 3
SELECTION_METRIC = 'Mean_OOF_AUC'
random.seed(SEED)
np.random.seed(SEED)

def sha256(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

protocol = {
    'artifact_id': 'recent_same_split_comparator_protocol',
    'purpose': 'Same-split comparison of recent representative approaches',
    'comparators': {
        'Pang_2025': 'MARS using R earth; degree 1 and degree 2 included in search',
        'Jose_2024': 'LightGBM using Python lightgbm.LGBMClassifier',
    },
    'interpretation': 'representative method reimplementation, not exact paper replication',
    'dataset_rows': 69057,
    'training_rows': 55245,
    'test_rows': 13812,
    'feature_count': 21,
    'folds': 3,
    'selection_metric': SELECTION_METRIC,
    'tie_breakers': ['Mean_OOF_F1', 'Mean_OOF_Recall', 'Configuration_ID'],
    'candidate_configurations_per_method': 20,
    'test_threshold': THRESHOLD,
    'seed': SEED,
    'data_sha256': sha256(DATA_FILE),
    'split_sha256': sha256(SPLIT_FILE),
    'test_used_for_tuning': False,
}
(WORK_ROOT / 'config/recent_comparators/frozen_protocol.json').write_text(json.dumps(protocol, indent=2) + '\n')
print(json.dumps(protocol, indent=2))



## Step 4 - Load and verify the existing prepared dataset and split



In [ ]:
data = pd.read_csv(DATA_FILE)
splits = pd.read_csv(SPLIT_FILE)

assert data.shape == (69057, 22), data.shape
assert splits.shape[0] == 69057
assert 'Outcome' in data.columns
assert splits['Source_Row_Index'].is_unique

data = data.copy()
data.insert(0, 'Source_Row_Index', np.arange(len(data), dtype=int))
linked = data.merge(splits, on='Source_Row_Index', how='inner', validate='one_to_one')
assert len(linked) == 69057

feature_cols = [c for c in data.columns if c not in ['Source_Row_Index', 'Outcome']]
assert len(feature_cols) == 21, feature_cols

train_df = linked[linked['Outer_Split'].eq('train')].sort_values('Train_Position').reset_index(drop=True)
test_df = linked[linked['Outer_Split'].eq('test')].sort_values('Test_Position').reset_index(drop=True)
assert len(train_df) == 55245
assert len(test_df) == 13812
assert set(train_df['OOF_Validation_Fold'].astype(int)) == {1, 2, 3}
assert test_df['OOF_Validation_Fold'].isna().all()

X_train = train_df[feature_cols].astype(float)
y_train = train_df['Outcome'].astype(int).to_numpy()
train_folds = train_df['OOF_Validation_Fold'].astype(int).to_numpy()
X_test = test_df[feature_cols].astype(float)
y_test = test_df['Outcome'].astype(int).to_numpy()

print('Training:', X_train.shape, 'Test:', X_test.shape)
print('Training outcome counts:', np.bincount(y_train))
print('Test outcome counts:', np.bincount(y_test))

# Save standardized inputs for the R MARS stage. These contain no new split.
mars_train = train_df[['Source_Row_Index', 'Train_Position', 'OOF_Validation_Fold', 'Outcome'] + feature_cols]
mars_test = test_df[['Source_Row_Index', 'Test_Position', 'Outcome'] + feature_cols]
mars_train.to_csv(WORK_ROOT / 'intermediate/recent_comparators/mars_train_input.csv', index=False)
mars_test.to_csv(WORK_ROOT / 'intermediate/recent_comparators/mars_test_input.csv', index=False)



## Step 5 - Freeze equal 20-configuration search spaces



In [ ]:
# Pang search: 2 interaction degrees x 5 maximum-term levels x 2 penalties = 20.
mars_grid = pd.DataFrame([
    {'Configuration_ID': f'MARS_{i:02d}', 'degree': degree, 'nk': nk, 'penalty': penalty}
    for i, (degree, nk, penalty) in enumerate(
        [(d, n, p) for d in [1, 2] for n in [15, 25, 35, 45, 55] for p in [2.0, 3.0]],
        start=1,
    )
])

# Jose search: 20 predefined LightGBM configurations, fixed before results.
lightgbm_grid = [
    {'num_leaves': 15, 'max_depth': 5, 'learning_rate': .03, 'n_estimators': 300, 'min_child_samples': 20, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_lambda': 0.0},
    {'num_leaves': 15, 'max_depth': 5, 'learning_rate': .05, 'n_estimators': 200, 'min_child_samples': 30, 'subsample': .9, 'colsample_bytree': .9, 'reg_lambda': 0.0},
    {'num_leaves': 15, 'max_depth': 6, 'learning_rate': .07, 'n_estimators': 150, 'min_child_samples': 40, 'subsample': .8, 'colsample_bytree': 1.0, 'reg_lambda': 1.0},
    {'num_leaves': 23, 'max_depth': 5, 'learning_rate': .03, 'n_estimators': 400, 'min_child_samples': 20, 'subsample': .9, 'colsample_bytree': .8, 'reg_lambda': 1.0},
    {'num_leaves': 23, 'max_depth': 6, 'learning_rate': .05, 'n_estimators': 250, 'min_child_samples': 30, 'subsample': 1.0, 'colsample_bytree': .9, 'reg_lambda': 0.0},
    {'num_leaves': 23, 'max_depth': 7, 'learning_rate': .07, 'n_estimators': 175, 'min_child_samples': 40, 'subsample': .8, 'colsample_bytree': .8, 'reg_lambda': 2.0},
    {'num_leaves': 31, 'max_depth': 5, 'learning_rate': .03, 'n_estimators': 400, 'min_child_samples': 20, 'subsample': .8, 'colsample_bytree': 1.0, 'reg_lambda': 0.0},
    {'num_leaves': 31, 'max_depth': 6, 'learning_rate': .05, 'n_estimators': 250, 'min_child_samples': 30, 'subsample': .9, 'colsample_bytree': .9, 'reg_lambda': 1.0},
    {'num_leaves': 31, 'max_depth': 7, 'learning_rate': .07, 'n_estimators': 175, 'min_child_samples': 50, 'subsample': 1.0, 'colsample_bytree': .8, 'reg_lambda': 2.0},
    {'num_leaves': 31, 'max_depth': -1, 'learning_rate': .05, 'n_estimators': 200, 'min_child_samples': 20, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_lambda': 0.0},
    {'num_leaves': 47, 'max_depth': 6, 'learning_rate': .03, 'n_estimators': 400, 'min_child_samples': 30, 'subsample': .8, 'colsample_bytree': .9, 'reg_lambda': 1.0},
    {'num_leaves': 47, 'max_depth': 7, 'learning_rate': .05, 'n_estimators': 250, 'min_child_samples': 40, 'subsample': .9, 'colsample_bytree': .8, 'reg_lambda': 2.0},
    {'num_leaves': 47, 'max_depth': 8, 'learning_rate': .07, 'n_estimators': 175, 'min_child_samples': 50, 'subsample': 1.0, 'colsample_bytree': .9, 'reg_lambda': 3.0},
    {'num_leaves': 63, 'max_depth': 6, 'learning_rate': .03, 'n_estimators': 400, 'min_child_samples': 40, 'subsample': .9, 'colsample_bytree': 1.0, 'reg_lambda': 2.0},
    {'num_leaves': 63, 'max_depth': 7, 'learning_rate': .05, 'n_estimators': 250, 'min_child_samples': 50, 'subsample': .8, 'colsample_bytree': .9, 'reg_lambda': 3.0},
    {'num_leaves': 63, 'max_depth': 8, 'learning_rate': .07, 'n_estimators': 175, 'min_child_samples': 60, 'subsample': 1.0, 'colsample_bytree': .8, 'reg_lambda': 4.0},
    {'num_leaves': 31, 'max_depth': 8, 'learning_rate': .02, 'n_estimators': 500, 'min_child_samples': 60, 'subsample': .8, 'colsample_bytree': .8, 'reg_lambda': 4.0},
    {'num_leaves': 47, 'max_depth': -1, 'learning_rate': .02, 'n_estimators': 500, 'min_child_samples': 80, 'subsample': .9, 'colsample_bytree': .9, 'reg_lambda': 5.0},
    {'num_leaves': 63, 'max_depth': -1, 'learning_rate': .03, 'n_estimators': 350, 'min_child_samples': 80, 'subsample': .8, 'colsample_bytree': 1.0, 'reg_lambda': 5.0},
    {'num_leaves': 95, 'max_depth': -1, 'learning_rate': .03, 'n_estimators': 300, 'min_child_samples': 100, 'subsample': .9, 'colsample_bytree': .8, 'reg_lambda': 6.0},
]
lightgbm_grid = pd.DataFrame([
    {'Configuration_ID': f'LGBM_{i:02d}', **params}
    for i, params in enumerate(lightgbm_grid, start=1)
])
assert len(mars_grid) == len(lightgbm_grid) == 20
mars_grid.to_csv(WORK_ROOT / 'config/recent_comparators/mars_search_space.csv', index=False)
lightgbm_grid.to_csv(WORK_ROOT / 'config/recent_comparators/lightgbm_search_space.csv', index=False)
print('Frozen configurations:', len(mars_grid), 'MARS and', len(lightgbm_grid), 'LightGBM')



## Step 6 - Common metric function



In [ ]:
def metric_row(model, y, probability, threshold=THRESHOLD, scope='test'):
    probability = np.asarray(probability, dtype=float)
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp)
    return {
        'Model': model, 'Scope': scope, 'Threshold': threshold,
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'Accuracy': accuracy_score(y, prediction),
        'Recall': recall_score(y, prediction, zero_division=0),
        'Specificity': specificity,
        'Precision': precision_score(y, prediction, zero_division=0),
        'F1': f1_score(y, prediction, zero_division=0),
        'AUC': roc_auc_score(y, probability),
        'Average_Precision': average_precision_score(y, probability),
        'Kappa': cohen_kappa_score(y, prediction),
        'MCC': matthews_corrcoef(y, prediction),
        'Brier': brier_score_loss(y, probability),
    }

def select_best(search_results):
    return search_results.sort_values(
        ['Mean_OOF_AUC', 'Mean_OOF_F1', 'Mean_OOF_Recall', 'Configuration_ID'],
        ascending=[False, False, False, True],
    ).iloc[0]



## Step 7 - Jose LightGBM: equal-budget three-fold search



In [ ]:
lgbm_search_rows = []
lgbm_start = time.time()

for _, cfg in lightgbm_grid.iterrows():
    fold_rows = []
    for fold in [1, 2, 3]:
        fit_mask = train_folds != fold
        val_mask = train_folds == fold
        params = cfg.drop('Configuration_ID').to_dict()
        for integer_param in ['num_leaves', 'max_depth', 'n_estimators', 'min_child_samples']:
            params[integer_param] = int(params[integer_param])
        model = LGBMClassifier(
            objective='binary', random_state=SEED, n_jobs=-1,
            deterministic=True, force_col_wise=True, verbosity=-1, **params
        )
        model.fit(X_train.loc[fit_mask], y_train[fit_mask])
        prob = model.predict_proba(X_train.loc[val_mask])[:, 1]
        fold_rows.append(metric_row(cfg.Configuration_ID, y_train[val_mask], prob, scope=f'OOF_Fold_{fold}'))
    fold_frame = pd.DataFrame(fold_rows)
    lgbm_search_rows.append({
        'Configuration_ID': cfg.Configuration_ID,
        'Mean_OOF_AUC': fold_frame.AUC.mean(),
        'Mean_OOF_F1': fold_frame.F1.mean(),
        'Mean_OOF_Recall': fold_frame.Recall.mean(),
        'SD_OOF_AUC': fold_frame.AUC.std(ddof=1),
    })
    print('Completed', cfg.Configuration_ID)

lgbm_search = pd.DataFrame(lgbm_search_rows)
lgbm_search.to_csv(WORK_ROOT / 'results/recent_comparators/lightgbm_search_results.csv', index=False)
best_lgbm = select_best(lgbm_search)
print('Selected LightGBM:', best_lgbm.to_dict())

# Generate locked OOF probabilities for the selected configuration.
best_cfg = lightgbm_grid.loc[
    lightgbm_grid['Configuration_ID'].eq(best_lgbm['Configuration_ID'])
].iloc[0]
best_params = best_cfg.drop('Configuration_ID').to_dict()
for integer_param in ['num_leaves', 'max_depth', 'n_estimators', 'min_child_samples']:
    best_params[integer_param] = int(best_params[integer_param])

lgbm_oof_prob = np.full(len(train_df), np.nan)
for fold in [1, 2, 3]:
    fit_mask, val_mask = train_folds != fold, train_folds == fold
    model = LGBMClassifier(
        objective='binary', random_state=SEED, n_jobs=-1,
        deterministic=True, force_col_wise=True, verbosity=-1, **best_params
    )
    model.fit(X_train.loc[fit_mask], y_train[fit_mask])
    lgbm_oof_prob[val_mask] = model.predict_proba(X_train.loc[val_mask])[:, 1]
assert np.isfinite(lgbm_oof_prob).all()

# Refit on all training rows and evaluate test once.
final_lgbm = LGBMClassifier(
    objective='binary', random_state=SEED, n_jobs=-1,
    deterministic=True, force_col_wise=True, verbosity=-1, **best_params
)
final_lgbm.fit(X_train, y_train)
lgbm_test_prob = final_lgbm.predict_proba(X_test)[:, 1]
joblib.dump(final_lgbm, WORK_ROOT / 'models/recent_comparators/jose_lightgbm.joblib')



## Step 8 - Pang MARS: equal-budget three-fold search in maintained R earth



In [ ]:
MARS_R_SCRIPT = r'''
suppressPackageStartupMessages(library(earth))

args <- commandArgs(trailingOnly=TRUE)
work_root <- args[1]
train_file <- file.path(work_root, 'intermediate', 'recent_comparators', 'mars_train_input.csv')
test_file <- file.path(work_root, 'intermediate', 'recent_comparators', 'mars_test_input.csv')
grid_file <- file.path(work_root, 'config', 'recent_comparators', 'mars_search_space.csv')

train <- read.csv(train_file, check.names=FALSE)
test <- read.csv(test_file, check.names=FALSE)
grid <- read.csv(grid_file, check.names=FALSE)
meta <- c('Source_Row_Index','Train_Position','Test_Position','OOF_Validation_Fold','Outcome')
features <- setdiff(names(train), meta)

scale_fit <- function(x) {
  mins <- sapply(x, min, na.rm=TRUE)
  maxs <- sapply(x, max, na.rm=TRUE)
  list(mins=mins, maxs=maxs)
}
scale_apply <- function(x, scaler) {
  ranges <- scaler$maxs - scaler$mins
  ranges[ranges == 0] <- 1
  out <- sweep(as.matrix(x), 2, scaler$mins, '-')
  out <- sweep(out, 2, ranges, '/')
  out * 2 - 1
}

auc_rank <- function(y, p) {
  pos <- p[y == 1]; neg <- p[y == 0]
  ranks <- rank(c(pos, neg), ties.method='average')
  n1 <- length(pos); n0 <- length(neg)
  (sum(ranks[seq_len(n1)]) - n1 * (n1 + 1) / 2) / (n1 * n0)
}
metrics <- function(y, p) {
  pred <- as.integer(p >= 0.5)
  tp <- sum(y == 1 & pred == 1); fn <- sum(y == 1 & pred == 0)
  fp <- sum(y == 0 & pred == 1); tn <- sum(y == 0 & pred == 0)
  recall <- tp / (tp + fn); precision <- tp / (tp + fp)
  f1 <- 2 * precision * recall / (precision + recall)
  c(AUC=auc_rank(y, p), F1=f1, Recall=recall)
}

search_rows <- list()
for (i in seq_len(nrow(grid))) {
  cfg <- grid[i,]
  fold_metrics <- list()
  for (fold in 1:3) {
    fit <- train$OOF_Validation_Fold != fold
    val <- train$OOF_Validation_Fold == fold
    scaler <- scale_fit(train[fit, features, drop=FALSE])
    xfit <- as.data.frame(scale_apply(train[fit, features, drop=FALSE], scaler))
    xval <- as.data.frame(scale_apply(train[val, features, drop=FALSE], scaler))
    names(xfit) <- features; names(xval) <- features
    fit_data <- cbind(Outcome=train$Outcome[fit], xfit)
    model <- earth(
      Outcome ~ ., data=fit_data, degree=cfg$degree, nk=cfg$nk,
      penalty=cfg$penalty, glm=list(family=binomial()), trace=0
    )
    prob <- as.numeric(predict(model, newdata=xval, type='response'))
    fold_metrics[[fold]] <- metrics(train$Outcome[val], prob)
  }
  fm <- do.call(rbind, fold_metrics)
  search_rows[[i]] <- data.frame(
    Configuration_ID=cfg$Configuration_ID,
    Mean_OOF_AUC=mean(fm[,'AUC']), Mean_OOF_F1=mean(fm[,'F1']),
    Mean_OOF_Recall=mean(fm[,'Recall']), SD_OOF_AUC=sd(fm[,'AUC'])
  )
  cat('Completed', cfg$Configuration_ID, '\n')
}
search <- do.call(rbind, search_rows)
write.csv(search, file.path(work_root, 'results', 'recent_comparators', 'mars_search_results.csv'), row.names=FALSE)
ordered <- search[order(-search$Mean_OOF_AUC, -search$Mean_OOF_F1,
                        -search$Mean_OOF_Recall, search$Configuration_ID),]
best_id <- ordered$Configuration_ID[1]
best <- grid[grid$Configuration_ID == best_id,][1,]

# Locked OOF predictions from selected configuration.
oof_prob <- rep(NA_real_, nrow(train))
for (fold in 1:3) {
  fit <- train$OOF_Validation_Fold != fold
  val <- train$OOF_Validation_Fold == fold
  scaler <- scale_fit(train[fit, features, drop=FALSE])
  xfit <- as.data.frame(scale_apply(train[fit, features, drop=FALSE], scaler))
  xval <- as.data.frame(scale_apply(train[val, features, drop=FALSE], scaler))
  names(xfit) <- features; names(xval) <- features
  model <- earth(Outcome ~ ., data=cbind(Outcome=train$Outcome[fit], xfit),
                 degree=best$degree, nk=best$nk, penalty=best$penalty,
                 glm=list(family=binomial()), trace=0)
  oof_prob[val] <- as.numeric(predict(model, newdata=xval, type='response'))
}
oof <- data.frame(
  Source_Row_Index=train$Source_Row_Index, Train_Position=train$Train_Position,
  OOF_Validation_Fold=train$OOF_Validation_Fold, y_true=train$Outcome,
  Predicted_Probability=oof_prob, Predicted_Class=as.integer(oof_prob >= 0.5),
  Model='Pang-MARS'
)
write.csv(oof, file.path(work_root, 'predictions', 'recent_comparators', 'pang_mars_oof_predictions.csv'), row.names=FALSE)

# Final training-only scaler and model; apply unchanged to test.
final_scaler <- scale_fit(train[, features, drop=FALSE])
xtrain <- as.data.frame(scale_apply(train[, features, drop=FALSE], final_scaler))
xtest <- as.data.frame(scale_apply(test[, features, drop=FALSE], final_scaler))
names(xtrain) <- features; names(xtest) <- features
final_model <- earth(Outcome ~ ., data=cbind(Outcome=train$Outcome, xtrain),
                     degree=best$degree, nk=best$nk, penalty=best$penalty,
                     glm=list(family=binomial()), trace=0)
test_prob <- as.numeric(predict(final_model, newdata=xtest, type='response'))
test_out <- data.frame(
  Source_Row_Index=test$Source_Row_Index, Test_Position=test$Test_Position,
  y_true=test$Outcome, Predicted_Probability=test_prob,
  Predicted_Class=as.integer(test_prob >= 0.5), Model='Pang-MARS'
)
write.csv(test_out, file.path(work_root, 'predictions', 'recent_comparators', 'pang_mars_test_predictions.csv'), row.names=FALSE)
write.csv(best, file.path(work_root, 'results', 'recent_comparators', 'selected_mars_configuration.csv'), row.names=FALSE)
saveRDS(final_model, file.path(work_root, 'models', 'recent_comparators', 'pang_mars_earth.rds'))
saveRDS(final_scaler, file.path(work_root, 'models', 'recent_comparators', 'pang_mars_scaler.rds'))
'''

mars_script_path = WORK_ROOT / 'intermediate/recent_comparators/run_pang_mars.R'
mars_script_path.write_text(MARS_R_SCRIPT)
subprocess.run(['Rscript', str(mars_script_path), str(WORK_ROOT)], check=True)

mars_search = pd.read_csv(WORK_ROOT / 'results/recent_comparators/mars_search_results.csv')
best_mars = select_best(mars_search)
print('Selected MARS:', best_mars.to_dict())



## Step 9 - Save comparable OOF and test prediction files



In [ ]:
def prediction_frame(source, position, fold, y, probability, model, scope):
    out = pd.DataFrame({
        'Source_Row_Index': np.asarray(source, dtype=int),
        'y_true': np.asarray(y, dtype=int),
        'Predicted_Probability': np.asarray(probability, dtype=float),
        'Predicted_Class': (np.asarray(probability) >= THRESHOLD).astype(int),
        'Decision_Threshold': THRESHOLD,
        'Model': model,
        'Scope': scope,
    })
    if scope == 'OOF':
        out.insert(1, 'Train_Position', np.asarray(position, dtype=int))
        out.insert(2, 'OOF_Validation_Fold', np.asarray(fold, dtype=int))
    else:
        out.insert(1, 'Test_Position', np.asarray(position, dtype=int))
    return out

lgbm_oof = prediction_frame(
    train_df.Source_Row_Index, train_df.Train_Position, train_folds,
    y_train, lgbm_oof_prob, 'Jose-LightGBM', 'OOF'
)
lgbm_test = prediction_frame(
    test_df.Source_Row_Index, test_df.Test_Position, None,
    y_test, lgbm_test_prob, 'Jose-LightGBM', 'Test'
)
lgbm_oof.to_csv(WORK_ROOT / 'predictions/recent_comparators/jose_lightgbm_oof_predictions.csv.gz', index=False)
lgbm_test.to_csv(WORK_ROOT / 'predictions/recent_comparators/jose_lightgbm_test_predictions.csv.gz', index=False)

mars_oof = pd.read_csv(WORK_ROOT / 'predictions/recent_comparators/pang_mars_oof_predictions.csv')
mars_test = pd.read_csv(WORK_ROOT / 'predictions/recent_comparators/pang_mars_test_predictions.csv')
mars_oof['Decision_Threshold'] = THRESHOLD
mars_oof['Scope'] = 'OOF'
mars_test['Decision_Threshold'] = THRESHOLD
mars_test['Scope'] = 'Test'
mars_oof.to_csv(WORK_ROOT / 'predictions/recent_comparators/pang_mars_oof_predictions.csv.gz', index=False)
mars_test.to_csv(WORK_ROOT / 'predictions/recent_comparators/pang_mars_test_predictions.csv.gz', index=False)



## Step 10 - Calculate identical performance metrics



In [ ]:
performance = pd.DataFrame([
    metric_row('Jose-LightGBM', y_train, lgbm_oof_prob, scope='OOF'),
    metric_row('Jose-LightGBM', y_test, lgbm_test_prob, scope='Test'),
    metric_row('Pang-MARS', mars_oof.y_true, mars_oof.Predicted_Probability, scope='OOF'),
    metric_row('Pang-MARS', mars_test.y_true, mars_test.Predicted_Probability, scope='Test'),
])
performance.to_csv(WORK_ROOT / 'results/recent_comparators/recent_comparator_performance_full_precision.csv', index=False)
display(performance)

selected = {
    'Jose-LightGBM': {
        'configuration_id': best_lgbm.Configuration_ID,
        'parameters': best_params,
    },
    'Pang-MARS': {
        'configuration_id': best_mars.Configuration_ID,
        'parameters': pd.read_csv(WORK_ROOT / 'results/recent_comparators/selected_mars_configuration.csv').iloc[0].to_dict(),
    },
}
(WORK_ROOT / 'results/recent_comparators/selected_configurations.json').write_text(json.dumps(selected, indent=2) + '\n')



## Step 11 - Structural verification before statistical comparison



In [ ]:
checks = []
def check(name, condition, observed, expected):
    checks.append({
        'Check': name, 'Observed': str(observed), 'Expected': str(expected),
        'Status': 'PASS' if condition else 'FAIL'
    })

for name, frame, expected_rows, key in [
    ('Jose LightGBM OOF', lgbm_oof, 55245, ['Source_Row_Index']),
    ('Jose LightGBM test', lgbm_test, 13812, ['Source_Row_Index']),
    ('Pang MARS OOF', mars_oof, 55245, ['Source_Row_Index']),
    ('Pang MARS test', mars_test, 13812, ['Source_Row_Index']),
]:
    check(f'{name} row count', len(frame) == expected_rows, len(frame), expected_rows)
    check(f'{name} unique source rows', not frame.duplicated(key).any(), frame.duplicated(key).sum(), 0)
    check(f'{name} probability range', frame.Predicted_Probability.between(0, 1).all(),
          [frame.Predicted_Probability.min(), frame.Predicted_Probability.max()], '[0,1]')
    expected_class = (frame.Predicted_Probability >= THRESHOLD).astype(int)
    check(f'{name} threshold agreement', np.array_equal(frame.Predicted_Class, expected_class),
          int((frame.Predicted_Class != expected_class).sum()), 0)

check('MARS search budget', len(mars_search) == 20, len(mars_search), 20)
check('LightGBM search budget', len(lgbm_search) == 20, len(lgbm_search), 20)
check('Performance result rows', len(performance) == 4, len(performance), 4)

verification = pd.DataFrame(checks)
verification.to_csv(WORK_ROOT / 'results/recent_comparators/recent_comparator_verification_checks.csv', index=False)
display(verification)
failed = verification.query("Status != 'PASS'")
if len(failed):
    raise AssertionError(f'{len(failed)} verification checks failed')
print(f'All {len(verification)} structural checks passed.')



## Step 12 - Save environment, file checksums and completion manifest



In [ ]:
environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scipy': scipy.__version__,
    'scikit_learn': sklearn.__version__,
    'lightgbm': lgb.__version__,
}
(WORK_ROOT / 'environment/recent_comparators/python_environment.json').write_text(json.dumps(environment, indent=2) + '\n')
subprocess.run(
    ['Rscript', '-e', f"writeLines(capture.output(sessionInfo()), '{WORK_ROOT / 'environment/recent_comparators/r_session_info.txt'}')"],
    check=True,
)

artifact_files = sorted(
    p for p in WORK_ROOT.rglob('*')
    if p.is_file() and p.name != 'artifact_checksums.csv'
)
checksum_frame = pd.DataFrame([
    {'Relative_Path': str(p.relative_to(WORK_ROOT)), 'Size_Bytes': p.stat().st_size, 'SHA256': sha256(p)}
    for p in artifact_files
])
checksum_frame.to_csv(WORK_ROOT / 'results/recent_comparators/artifact_checksums.csv', index=False)

manifest = {
    'status': 'completed',
    'verification_passed': int((verification.Status == 'PASS').sum()),
    'verification_failed': int((verification.Status != 'PASS').sum()),
    'models': ['Pang-MARS', 'Jose-LightGBM'],
    'threshold': THRESHOLD,
    'test_used_for_tuning': False,
    'interpretation': 'same-dataset, same-split head-to-head implementation of representative recent methods',
    'not_exact_paper_replication': True,
}
(WORK_ROOT / 'results/recent_comparators/recent_comparator_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print(json.dumps(manifest, indent=2))



## Verification

After the comparator workflow completes, run `tests/verify_all_results.py`
from the repository root to verify the generated predictions and statistical
outputs against the frozen release protocol.
